# Da base à tabela

Como exemplo, este notebook usa uma base com 22.866 sentenças do TJSP em ações contra bancos. O cliente quer saber em quantas o pedido foi negado, e por quê. Este notebook faz o caminho inteiro numa amostra pequena: carregar a base, estimar o custo, pedir ao modelo uma resposta no formato de uma tabela e conferir se ele acertou.

O arquivo `dataset_clinica_20261.csv`, com o link no site da aula, precisa estar na mesma pasta deste notebook. No Colab, abra o painel "Arquivos" (o ícone de pasta na barra lateral) e arraste o arquivo para lá; ele some quando a sessão termina.

## 1. Instalar

O `dataframeit` aplica um modelo de linguagem a cada linha de uma tabela. O extra `groq` instala a integração com o Groq, que tem plano gratuito.

In [ ]:
%pip install "dataframeit[groq]" pandas python-dotenv

## 2. A chave do Groq

1. Crie uma conta em https://console.groq.com e gere uma chave em "API Keys".
2. Na pasta deste notebook, crie um arquivo chamado `.env` com uma linha só: `GROQ_API_KEY=sua_chave_aqui`.
3. Não compartilhe o `.env` nem cole a chave numa célula. Quem tiver a chave usa a sua conta.

No Colab não há `.env`: use o ícone de chave na barra lateral ("Secrets"), crie o segredo `GROQ_API_KEY` com a sua chave e ligue o acesso deste notebook. A célula abaixo funciona nos dois lugares.

In [ ]:
import os

try:
    # No Colab, a chave vem do segredo GROQ_API_KEY.
    from google.colab import userdata
    os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")
except ImportError:
    # No computador, vem do arquivo .env.
    from dotenv import load_dotenv
    load_dotenv()

# Confere se a chave foi lida, sem mostrar a chave.
print("chave encontrada" if os.getenv("GROQ_API_KEY") else "chave não encontrada")

## 3. Uma pergunta ao modelo

A chave não aparece no código: o `ChatGroq` a lê do ambiente.

In [ ]:
from langchain_groq import ChatGroq

modelo = ChatGroq(model="openai/gpt-oss-120b")
resposta = modelo.invoke("Em uma frase: o que é uma sentença de improcedência?")
print(resposta.content)

## 4. A base e o custo

Cada 4 caracteres valem cerca de 1 token. A base tem textos de decisão repetidos: eles saem antes de qualquer conta, porque mandar o mesmo texto duas vezes custa duas vezes.

In [ ]:
import pandas as pd

sentencas = pd.read_csv("dataset_clinica_20261.csv")
print(len(sentencas), "linhas")

sentencas = sentencas.drop_duplicates(subset="decisao")
print(len(sentencas), "textos distintos")

sentencas["tokens"] = sentencas["decisao"].str.len() / 4
print(round(sentencas["tokens"].sum() / 1_000_000, 1), "milhões de tokens")

In [ ]:
# Preço de entrada por milhão de tokens, em dólares: o do openai/gpt-oss-120b no plano pago do Groq em
# 05/10/2026. Confira o preço do dia antes de usar.
preco_entrada = 0.15
custo = sentencas["tokens"].sum() / 1_000_000 * preco_entrada
print("US$", round(custo, 2), "só de entrada, para a base inteira")

## 5. Uma amostra que cabe no plano gratuito

O plano gratuito aceita 8 mil tokens por minuto. Uma sentença maior que isso não passa sozinha, e por isso a amostra fica só com as de até 6 mil tokens. Com semente fixa, rodar de novo sorteia as mesmas sentenças.

In [ ]:
amostra = sentencas[sentencas["tokens"] <= 6000].sample(n=10, random_state=2026)
amostra[["assunto", "tokens"]]

## 6. A tabela que queremos de volta

Cada campo da classe vira uma coluna. `Literal` obriga o modelo a escolher uma das opções; `list[Literal]` deixa escolher nenhuma, uma ou várias; `bool` pede `True` ou `False`. A `description` de cada campo vai ao modelo e diz o que coletar: é ali que se explica o que conta como cada opção. Esta é a classe já corrigida depois da conferência da aula.

In [ ]:
from typing import Literal
from pydantic import BaseModel, Field


class Decisao(BaseModel):
    resultado: Literal[
        "procedente", "parcialmente procedente",
        "improcedente", "extinto sem resolução do mérito",
        "acordo homologado", "outro"] = Field(
        description="Resultado do pedido principal, lido no dispositivo. Acordo homologado é só "
                    "acordo entre as partes; desistência homologada é extinção sem resolução do "
                    "mérito. Um pedido acolhido e outro negado é parcialmente procedente.")
    motivos: list[Literal[
        "contratação comprovada",
        "encargos sem abusividade",
        "golpe por culpa da vítima ou de terceiro",
        "falta de prova mínima",
        "prescrição ou decadência",
        "outro"]] = Field(
        description="Por que o pedido foi negado. Lista vazia se o resultado não for improcedente.")
    ma_fe: bool = Field(
        description="True só se a sentença condenou alguma parte por litigância de má-fé; "
                    "citar a má-fé para afastá-la é False.")

O prompt diz a tarefa; o que conta como cada opção já está nas descriptions. `{texto}` é trocado pelo texto de cada linha.

In [ ]:
prompt = """Você lê sentenças de ações contra bancos e preenche os campos pedidos.

Sentença:
{texto}
"""

## 7. Rodar

`rate_limit_delay` espera alguns segundos entre uma sentença e outra, para não passar do limite por minuto do plano gratuito. As dez levam alguns minutos.

In [ ]:
from dataframeit import dataframeit

tabela = dataframeit(amostra, Decisao, prompt,
                     provider="groq", model="openai/gpt-oss-120b",
                     text_column="decisao", rate_limit_delay=15)
tabela[["resultado", "motivos", "ma_fe"]]

In [ ]:
print(tabela["resultado"].value_counts())

## 8. O modelo acertou?

Leia o dispositivo de cada sentença, que a célula abaixo mostra a partir do último "ante o exposto" ou "posto isso", e anote o resultado na lista `leitura`, na mesma ordem da tabela. Use exatamente as opções da classe. Se o trecho mostrado não trouxer o dispositivo, leia a sentença inteira em `tabela["decisao"][numero]`.

In [ ]:
import re

# O dispositivo costuma vir depois do último "ante o exposto" ou "posto isso"; o fim da sentença
# é assinatura e jurisprudência citada, e não serve para a leitura.
for numero, texto in enumerate(tabela["decisao"]):
    fechos = list(re.finditer(r"exposto|posto isso|isto posto", texto, re.IGNORECASE))
    inicio = fechos[-1].start() if fechos else max(0, len(texto) - 900)
    print(numero, texto[inicio:inicio + 700])
    print("-" * 60)

In [ ]:
leitura = [
    # "improcedente",
]

In [ ]:
if len(leitura) == len(tabela):
    confere = tabela["resultado"] == leitura
    print(confere.sum(), "de", len(tabela))
    print(tabela.loc[~confere, ["resultado"]].assign(lido=[l for l, c in zip(leitura, confere) if not c]))
else:
    print("Anote", len(tabela), "resultados em leitura.")

Uma concordância numa amostra estima a base inteira e não a garante: as linhas que ninguém leu continuam sem conferência. Se o erro tiver padrão, ajuste o prompt e confira de novo numa amostra nova.

## 9. Guardar

O resultado vai para um CSV, para não pagar de novo por uma resposta que já veio.

In [ ]:
tabela.to_csv("resultados_amostra.csv", index=False, encoding="utf-8-sig")